# Scrapy 爬虫框架入门

Scrapy 是一个强大的 Python 爬虫框架，适合大规模数据爬取。

## 目录
1. Scrapy 简介
2. 项目结构
3. Spider 编写
4. Item 定义
5. Pipeline 处理
6. 中间件使用

## 1. Scrapy 简介

### 安装
```bash
pip install scrapy
```

### 创建项目
```bash
scrapy startproject myproject
cd myproject
scrapy genspider example example.com
```

### Scrapy 架构
- Engine: 引擎，控制数据流
- Scheduler: 调度器，管理请求队列
- Downloader: 下载器，下载网页
- Spider: 爬虫，解析响应
- Pipeline: 管道，处理数据
- Middleware: 中间件，处理请求和响应

## 2. 项目结构

```
myproject/
    scrapy.cfg              # 配置文件
    myproject/
        __init__.py
        items.py            # 数据模型
        middlewares.py      # 中间件
        pipelines.py        # 数据管道
        settings.py         # 设置
        spiders/            # 爬虫目录
            __init__.py
            example.py      # 爬虫文件
```

## 3. Spider 编写

### 基础 Spider

In [ ]:
# 这是一个示例代码，展示 Scrapy Spider 的结构
# 实际运行需要在 Scrapy 项目中

"""
import scrapy

class ExampleSpider(scrapy.Spider):
    name = 'example'  # 爬虫名称
    allowed_domains = ['example.com']  # 允许的域名
    start_urls = ['http://example.com/']  # 起始 URL
    
    def parse(self, response):
        # 解析响应
        title = response.css('title::text').get()
        yield {
            'title': title,
            'url': response.url
        }
"""

print("基础 Spider 结构示例")
print("="*50)
print("主要组成部分:")
print("1. name: 爬虫的唯一标识")
print("2. start_urls: 起始 URL 列表")
print("3. parse(): 默认的响应处理方法")
print("4. yield: 返回数据或新的请求")

### 电影爬虫示例

In [ ]:
# 电影爬虫示例代码

"""
import scrapy
from myproject.items import MovieItem

class MovieSpider(scrapy.Spider):
    name = 'movie'
    allowed_domains = ['example.com']
    start_urls = ['http://example.com/movies']
    
    def parse(self, response):
        # 提取电影列表
        for movie in response.css('div.movie-item'):
            item = MovieItem()
            item['name'] = movie.css('h2.title::text').get()
            item['score'] = movie.css('span.score::text').get()
            item['year'] = movie.css('span.year::text').get()
            
            # 获取详情页链接
            detail_url = movie.css('a::attr(href)').get()
            if detail_url:
                yield response.follow(detail_url, 
                                     callback=self.parse_detail,
                                     meta={'item': item})
        
        # 翻页
        next_page = response.css('a.next::attr(href)').get()
        if next_page:
            yield response.follow(next_page, callback=self.parse)
    
    def parse_detail(self, response):
        # 解析详情页
        item = response.meta['item']
        item['director'] = response.css('span.director::text').get()
        item['actors'] = response.css('span.actor::text').getall()
        item['summary'] = response.css('div.summary::text').get()
        
        yield item
"""

print("电影爬虫示例")
print("="*50)
print("功能:")
print("1. 爬取电影列表")
print("2. 跟进详情页链接")
print("3. 自动翻页")
print("4. 使用 meta 传递数据")

## 4. Item 定义

Items 是数据容器，定义要爬取的字段

In [ ]:
# items.py 示例

"""
import scrapy

class MovieItem(scrapy.Item):
    # 定义字段
    name = scrapy.Field()       # 电影名称
    score = scrapy.Field()      # 评分
    year = scrapy.Field()       # 年份
    director = scrapy.Field()   # 导演
    actors = scrapy.Field()     # 演员列表
    summary = scrapy.Field()    # 简介
    url = scrapy.Field()        # URL

class ProxyItem(scrapy.Item):
    # 代理 IP 数据模型
    ip = scrapy.Field()
    port = scrapy.Field()
    protocol = scrapy.Field()
    location = scrapy.Field()
    speed = scrapy.Field()
"""

print("Item 定义示例")
print("="*50)
print("Item 的作用:")
print("1. 定义数据结构")
print("2. 提供字段验证")
print("3. 便于数据处理")
print("4. 类似于字典，但更规范")

## 5. Pipeline 处理

Pipeline 用于处理爬取的数据

In [ ]:
# pipelines.py 示例

"""
import json
import csv

class JsonPipeline:
    '''保存为 JSON 文件'''
    
    def open_spider(self, spider):
        self.file = open('output.json', 'w', encoding='utf-8')
        self.items = []
    
    def close_spider(self, spider):
        json.dump(self.items, self.file, 
                 ensure_ascii=False, indent=2)
        self.file.close()
    
    def process_item(self, item, spider):
        self.items.append(dict(item))
        return item

class CsvPipeline:
    '''保存为 CSV 文件'''
    
    def open_spider(self, spider):
        self.file = open('output.csv', 'w', 
                        encoding='utf-8', newline='')
        self.writer = None
    
    def close_spider(self, spider):
        self.file.close()
    
    def process_item(self, item, spider):
        if not self.writer:
            self.writer = csv.DictWriter(
                self.file, fieldnames=item.keys())
            self.writer.writeheader()
        
        self.writer.writerow(dict(item))
        return item

class DataCleanPipeline:
    '''数据清洗'''
    
    def process_item(self, item, spider):
        # 清理空白字符
        if item.get('name'):
            item['name'] = item['name'].strip()
        
        # 转换数据类型
        if item.get('score'):
            item['score'] = float(item['score'])
        
        # 去重检查
        # ...
        
        return item
"""

print("Pipeline 示例")
print("="*50)
print("Pipeline 的作用:")
print("1. 数据清洗和验证")
print("2. 保存到文件或数据库")
print("3. 去重")
print("4. 数据转换")
print("\n执行顺序由 settings.py 中的优先级决定")

## 6. 中间件使用

### 下载中间件

In [ ]:
# middlewares.py 示例

"""
import random
from scrapy import signals

class RandomUserAgentMiddleware:
    '''随机 User-Agent 中间件'''
    
    def __init__(self):
        self.user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
            'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'
        ]
    
    def process_request(self, request, spider):
        request.headers['User-Agent'] = random.choice(self.user_agents)

class ProxyMiddleware:
    '''代理中间件'''
    
    def __init__(self):
        self.proxies = [
            'http://proxy1:8080',
            'http://proxy2:8080',
        ]
    
    def process_request(self, request, spider):
        proxy = random.choice(self.proxies)
        request.meta['proxy'] = proxy
    
    def process_exception(self, request, exception, spider):
        # 代理失败时的处理
        spider.logger.warning(f'代理失败: {exception}')
        return request

class RetryMiddleware:
    '''重试中间件'''
    
    def process_response(self, request, response, spider):
        if response.status in [500, 502, 503, 504]:
            # 服务器错误，重试
            return request
        return response
"""

print("中间件示例")
print("="*50)
print("中间件类型:")
print("1. 下载中间件: 处理请求和响应")
print("2. Spider 中间件: 处理 Spider 的输入输出")
print("\n常用功能:")
print("- 随机 User-Agent")
print("- 代理 IP 轮换")
print("- 请求重试")
print("- Cookie 处理")

## 7. Settings 配置

In [ ]:
# settings.py 常用配置

"""
# 基础设置
BOT_NAME = 'myproject'
SPIDER_MODULES = ['myproject.spiders']
NEWSPIDER_MODULE = 'myproject.spiders'

# 遵守 robots.txt
ROBOTSTXT_OBEY = False

# 并发设置
CONCURRENT_REQUESTS = 16
CONCURRENT_REQUESTS_PER_DOMAIN = 8

# 下载延迟（秒）
DOWNLOAD_DELAY = 1

# Cookie
COOKIES_ENABLED = True

# User-Agent
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'

# 中间件配置
DOWNLOADER_MIDDLEWARES = {
    'myproject.middlewares.RandomUserAgentMiddleware': 400,
    'myproject.middlewares.ProxyMiddleware': 410,
}

# Pipeline 配置
ITEM_PIPELINES = {
    'myproject.pipelines.DataCleanPipeline': 100,
    'myproject.pipelines.JsonPipeline': 200,
    'myproject.pipelines.CsvPipeline': 300,
}

# 日志级别
LOG_LEVEL = 'INFO'
LOG_FILE = 'scrapy.log'

# 自动限速
AUTOTHROTTLE_ENABLED = True
AUTOTHROTTLE_START_DELAY = 1
AUTOTHROTTLE_MAX_DELAY = 10
"""

print("Scrapy 配置说明")
print("="*50)
print("重要配置项:")
print("1. CONCURRENT_REQUESTS: 并发请求数")
print("2. DOWNLOAD_DELAY: 下载延迟")
print("3. DOWNLOADER_MIDDLEWARES: 中间件")
print("4. ITEM_PIPELINES: 数据管道")
print("5. AUTOTHROTTLE: 自动限速")
print("\n数字越小，优先级越高")

## 8. 运行爬虫

### 命令行运行
```bash
# 运行爬虫
scrapy crawl movie

# 保存到文件
scrapy crawl movie -o output.json
scrapy crawl movie -o output.csv

# 查看所有爬虫
scrapy list

# 查看设置
scrapy settings --get BOT_NAME
```

### 代码运行

In [ ]:
# 在 Python 代码中运行 Scrapy

"""
from scrapy.crawler import CrawlerProcess
from scrapy.utils.project import get_project_settings

# 获取项目设置
settings = get_project_settings()

# 创建爬虫进程
process = CrawlerProcess(settings)

# 添加爬虫
process.crawl('movie')

# 启动
process.start()
"""

print("Scrapy 运行方式")
print("="*50)
print("1. 命令行: scrapy crawl spider_name")
print("2. Python 代码: CrawlerProcess")
print("3. Scrapyd: 部署和调度")

## 9. 实用技巧

### CSS 选择器

In [ ]:
# CSS 选择器示例

"""
# 获取文本
response.css('h1::text').get()
response.css('h1::text').getall()

# 获取属性
response.css('a::attr(href)').get()
response.css('img::attr(src)').getall()

# 类选择器
response.css('.movie-title::text').get()

# ID 选择器
response.css('#main-content::text').get()

# 组合选择器
response.css('div.movie h2.title::text').get()

# 属性选择器
response.css('a[data-id="123"]::attr(href)').get()
"""

print("CSS 选择器常用方法")
print("="*50)
print(".get()     - 获取第一个匹配")
print(".getall()  - 获取所有匹配")
print("::text     - 获取文本内容")
print("::attr()   - 获取属性值")

### XPath 选择器

In [ ]:
# XPath 选择器示例

"""
# 获取文本
response.xpath('//h1/text()').get()
response.xpath('//h1/text()').getall()

# 获取属性
response.xpath('//a/@href').get()
response.xpath('//img/@src').getall()

# 类选择
response.xpath('//div[@class="movie"]/h2/text()').get()

# ID 选择
response.xpath('//div[@id="content"]/text()').get()

# 包含文本
response.xpath('//a[contains(text(), "下一页")]/@href').get()

# 多条件
response.xpath('//div[@class="movie" and @data-id="123"]/text()').get()
"""

print("XPath 选择器常用语法")
print("="*50)
print("//        - 选择所有后代节点")
print("/         - 选择直接子节点")
print("@         - 选择属性")
print("text()    - 获取文本")
print("contains()- 包含判断")

## 练习题

1. 创建一个 Scrapy 项目，爬取豆瓣电影 Top250
2. 实现自动翻页功能
3. 添加随机 User-Agent 中间件
4. 将数据保存为 JSON 和 CSV 两种格式
5. 添加数据去重功能

## 最佳实践

- 合理设置下载延迟，避免被封
- 使用 Item 定义数据结构
- 在 Pipeline 中处理数据
- 使用中间件处理请求
- 记录日志，便于调试
- 遵守网站的 robots.txt